# Data to Metadata to Dummy Data

In [1]:
import pandas as pd
import json

In [2]:
df = pd.read_csv(
    "penguin_plus.csv", 
    parse_dates=["timestamp", "timestamp_with_time"]
)
df.head(2)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,Adelie,Torgersen,39.1,18.7,NaN,3750.0,True,73,2025-04-13,2025-04-13 15:04:00,1
1,Adelie,Torgersen,39.5,17.4,NaN,3800.0,False,1,2025-12-15,2025-12-15 18:15:26,2


In [3]:
import sys
sys.path.append('../')

In [4]:
from csvw_safe.make_metadata_from_data import make_metadata_from_data
from csvw_safe.validate_metadata import validate_metadata
from csvw_safe.validate_metadata_shacl import validate_metadata_shacl
from csvw_safe.make_dummy_from_metadata import make_dummy_from_metadata
from csvw_safe.assert_same_structure import assert_same_structure

In [5]:
shacl_path = '../../csvw-safe-constraints.ttl'

In [6]:
metadata_path_1 = 'metadata/penguin_metadata_basic.json-ld'

metadata_path_2 = 'metadata/penguin_metadata_continuous_partitions.json-ld'
metadata_path_3 = 'metadata/penguin_metadata_continuous_partitions_column-contrib.json-ld'
metadata_path_4 = 'metadata/penguin_metadata_continuous_partitions_partition-contrib.json-ld'

metadata_path_5 = 'metadata/penguin_metadata_column_groups.json-ld'

metadata_path_6 = 'metadata/penguin_metadata_continuous_partitions_in_column_groups.json-ld'
metadata_path_7 = 'metadata/penguin_metadata_continuous_partitions_in_column_groups_column-contrib.json-ld'
metadata_path_8 = 'metadata/penguin_metadata_continuous_partitions_in_column_groups_partition-contrib.json-ld'

## Basic

In [7]:
metadata_1 = make_metadata_from_data(
    df, privacy_unit = "penguin_id"
)
with open(metadata_path_1, 'w', encoding='utf-8') as f:
    json.dump(metadata_1, f)

In [8]:
errors = validate_metadata(metadata_1)
print(errors)
validate_metadata_shacl(metadata_path_1, shacl_path)

[]


(True, 'Validation Report\nConforms: True\n')

In [9]:
dummy_df_1 = make_dummy_from_metadata(metadata_1, nb_rows = 100, seed = 0)
dummy_df_1.head(2)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,<NA>,<NA>,49.616446,15.521978,182.0,3865,<NA>,160,2025-12-26,2025-05-05 06:47:46,2
1,<NA>,<NA>,NaN,18.023434,NaN,4444,<NA>,135,2025-03-25,2025-01-29 06:47:46,3


In [10]:
assert_same_structure(df, dummy_df_1, check_categories = False)

AssertionError: Column 'sex' dtype mismatch: original=boolean, dummy=string

## Known continuous partitions

In [10]:
continuous_partitions = {
    "bill_length_mm": [30.0, 40.0, 50.0, 60.0],
    "timestamp": ['2025-01-02 00:00:00', '2025-06-02 00:00:00', '2025-12-30 00:00:00'],
}

In [11]:
metadata_2 = make_metadata_from_data(
    df, 
    privacy_unit = "penguin_id", 
    continuous_partitions = continuous_partitions
)
with open(metadata_path_2, 'w', encoding='utf-8') as f:
    json.dump(metadata_2, f)

In [12]:
errors = validate_metadata(metadata_2)
print(errors)
validate_metadata_shacl(metadata_path_2, shacl_path)

[]


(True, 'Validation Report\nConforms: True\n')

In [13]:
dummy_df_2 = make_dummy_from_metadata(metadata_2, nb_rows = 100, seed = 0)
dummy_df_2.head(2)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,<NA>,<NA>,<NA>,18.563858,NaN,3697,<NA>,102,2025-04-29,2025-06-20 06:47:46,5
1,<NA>,<NA>,<NA>,18.882953,219.0,3158,<NA>,292,2025-06-26,2025-05-25 06:47:46,1


## With DP at column level

In [14]:
metadata_3 = make_metadata_from_data(
    df,
    privacy_unit = "penguin_id", 
    continuous_partitions = continuous_partitions,
    default_contributions_level = 'column'
)
with open(metadata_path_3, 'w', encoding='utf-8') as f:
    json.dump(metadata_3, f)

In [15]:
errors = validate_metadata(metadata_3)
print(errors)
validate_metadata_shacl(metadata_path_3, shacl_path)

[]


(True, 'Validation Report\nConforms: True\n')

In [16]:
dummy_df_3 = make_dummy_from_metadata(metadata_3, nb_rows = 100, seed = 0)
dummy_df_3.head(2)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,Gentoo,Dream,<NA>,15.521978,182.0,3865,False,338,2025-05-05,2025-06-24 06:47:46,2
1,Chinstrap,Torgersen,<NA>,18.023434,NaN,4444,False,78,2025-01-29,2025-07-29 06:47:46,2


## With DP at partition level

In [17]:
metadata_4 = make_metadata_from_data(
    df,
    privacy_unit = "penguin_id", 
    continuous_partitions = continuous_partitions,
    default_contributions_level = 'partition'
)
with open(metadata_path_4, 'w', encoding='utf-8') as f:
    json.dump(metadata_4, f)

In [18]:
errors = validate_metadata(metadata_4)
print(errors)
validate_metadata_shacl(metadata_path_4, shacl_path)

[]


(True, 'Validation Report\nConforms: True\n')

In [19]:
dummy_df_4 = make_dummy_from_metadata(metadata_4, nb_rows = 100, seed = 0)
dummy_df_4.head(2)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,Gentoo,Dream,<NA>,15.521978,182.0,3865,False,338,2025-05-05,2025-06-24 06:47:46,2
1,Chinstrap,Torgersen,<NA>,18.023434,NaN,4444,False,78,2025-01-29,2025-07-29 06:47:46,2


## Known column groups

In [20]:
column_groups = [
    ["species", "island"],
    ["species", "island", "favourite_number"]
]

In [21]:
metadata_5 = make_metadata_from_data(
    df,
    privacy_unit = "penguin_id", 
    column_groups = column_groups,
    default_contributions_level = 'column'
)
with open(metadata_path_5, 'w', encoding='utf-8') as f:
    json.dump(metadata_5, f)

In [22]:
errors = validate_metadata(metadata_5)
print(errors)
validate_metadata_shacl(metadata_path_5, shacl_path)

[]


(True, 'Validation Report\nConforms: True\n')

In [23]:
dummy_df_5 = make_dummy_from_metadata(metadata_5, nb_rows = 100, seed = 0)
dummy_df_5.head(2)

,species,island,favourite_number,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time
0,Adelie,Torgersen,0,45.299668,15.771346,NaN,6253,False,164,2025-06-14,2025-07-24 06:47:46
1,Chinstrap,Dream,5,38.490255,19.071391,212.0,3515,False,197,2025-06-22,2025-06-06 06:47:46


## Known column groups containing public continuous partitions

In [24]:
continuous_partitions = {
    "bill_length_mm": [30.0, 40.0, 50.0, 60.0],
    "timestamp": ['2025-01-02 00:00:00', '2025-06-02 00:00:00', '2025-12-30 00:00:00'],
}
column_groups = [
    ["species", "island"],
    ["species", "island", "bill_length_mm"],
]

In [25]:
metadata_6 = make_metadata_from_data(
    df, 
    privacy_unit = "penguin_id", 
    continuous_partitions = continuous_partitions,
    column_groups = column_groups,
    default_contributions_level = 'column'
)
with open(metadata_path_6, 'w', encoding='utf-8') as f:
    json.dump(metadata_6, f)

In [26]:
errors = validate_metadata(metadata_6)
print(errors)
validate_metadata_shacl(metadata_path_6, shacl_path)

[]


(True, 'Validation Report\nConforms: True\n')

In [27]:
dummy_df_6 = make_dummy_from_metadata(metadata_6, nb_rows = 100, seed = 0)
dummy_df_6.head(2)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,Adelie,Torgersen,<NA>,17.131899,NaN,4414,True,237,2025-12-20,2025-07-19 06:47:46,3
1,Chinstrap,Dream,<NA>,15.051933,193.0,2752,True,135,2025-01-12,2025-12-26 06:47:46,2


## Known column groups containing public continuous partitions - dp contrib column

In [28]:
metadata_7 = make_metadata_from_data(
    df, 
    privacy_unit = "penguin_id", 
    continuous_partitions = continuous_partitions,
    column_groups = column_groups,
    default_contributions_level = 'column'
)
with open(metadata_path_7, 'w', encoding='utf-8') as f:
    json.dump(metadata_7, f)

In [29]:
errors = validate_metadata(metadata_7)
print(errors)
validate_metadata_shacl(metadata_path_7, shacl_path)

[]


(True, 'Validation Report\nConforms: True\n')

In [30]:
dummy_df_7 = make_dummy_from_metadata(metadata_7, nb_rows = 100, seed = 0)
dummy_df_7.head(2)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,Adelie,Torgersen,<NA>,17.131899,NaN,4414,True,237,2025-12-20,2025-07-19 06:47:46,3
1,Chinstrap,Dream,<NA>,15.051933,193.0,2752,True,135,2025-01-12,2025-12-26 06:47:46,2


## Known column groups containing public continuous partitions - dp contrib partition

In [31]:
metadata_8 = make_metadata_from_data(
    df, 
    privacy_unit = "penguin_id", 
    continuous_partitions = continuous_partitions,
    column_groups = column_groups,
    default_contributions_level = 'partition'
)
with open(metadata_path_8, 'w', encoding='utf-8') as f:
    json.dump(metadata_8, f)

In [32]:
errors = validate_metadata(metadata_8)
print(errors)
validate_metadata_shacl(metadata_path_8, shacl_path)

[]


(True, 'Validation Report\nConforms: True\n')

In [33]:
dummy_df_8 = make_dummy_from_metadata(metadata_8, nb_rows = 100, seed = 0)
dummy_df_8.head(2)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,Adelie,Torgersen,<NA>,17.131899,NaN,4414,True,237,2025-12-20,2025-07-19 06:47:46,3
1,Chinstrap,Dream,<NA>,15.051933,193.0,2752,True,135,2025-01-12,2025-12-26 06:47:46,2
